<a href="https://colab.research.google.com/github/amit-sw/colab_notebooks/blob/main/MCP_StockQuotes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup

In [1]:
!pip install openai langchain python-dotenv toml --quiet

In [2]:
import os
import re
import json
import requests

from openai import OpenAI

In [3]:
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["ALPHAVANTAGE_API_KEY"] = userdata.get('ALPHAVANTAGE_API_KEY')
AI_MODEL = 'gpt-5-mini'

ALPHAVANTAGE_API_URL = "https://www.alphavantage.co/query"
ALPHAVANTAGE_API_KEY = os.environ["ALPHAVANTAGE_API_KEY"]

client=OpenAI(api_key=os.environ['OPENAI_API_KEY'])

# Direct API call

## Alphavantage

In [11]:
def get_stock_quote(symbol: str) -> dict:
    """Fetch the latest stock quote for ``symbol`` using the Alpha Vantage API."""
    params = {"function": "GLOBAL_QUOTE", "symbol": symbol, "apikey": ALPHAVANTAGE_API_KEY}
    response = requests.get(ALPHAVANTAGE_API_URL, params=params, timeout=10)
    response.raise_for_status()
    data = response.json().get("Global Quote", {})
    if not data:
        return {"symbol": symbol.upper(), "price": None}
    return {
        "symbol": data.get("01. symbol", symbol.upper()),
        "price": float(data.get("05. price", "0") or 0.0),
        "change_percent": data.get("10. change percent"),
    }

## Yahoo Finance

In [5]:
!pip install yfinance --quiet
import yfinance as yf
import json

In [6]:
def get_stock_quote_yf(symbol: str) -> str:
    """Get current stock price and basic information."""
    try:
        stock = yf.Ticker(symbol)
        info = stock.info
        hist = stock.history(period="1d")
        if hist.empty:
            return json.dumps({"error": f"Could not retrieve data for {symbol}"})

        current_price = hist['Close'].iloc[-1]
        result = {
            "symbol": symbol,
            "current_price": round(current_price, 2),
            "company_name": info.get('longName', symbol),
            "market_cap": info.get('marketCap', 0),
            "pe_ratio": info.get('trailingPE', 'N/A'),
            "52_week_high": info.get('fiftyTwoWeekHigh', 0),
            "52_week_low": info.get('fiftyTwoWeekLow', 0)
        }
        return json.dumps(result, indent=2)

    except Exception as e:
        return json.dumps({"error": str(e)})

# Get quotes by calling API

In [14]:
get_stock_quote("TT")

{'symbol': 'TT', 'price': 479.12, 'change_percent': '0.3939%'}

In [15]:
get_stock_quote_yf("TT")

'{\n  "symbol": "TT",\n  "current_price": 479.12,\n  "company_name": "Trane Technologies plc",\n  "market_cap": 106239590400,\n  "pe_ratio": 36.49048,\n  "52_week_high": 484.9,\n  "52_week_low": 334.37\n}'

# GenAI without MCP

In [16]:
def get_completion(model, messages):
    print(f"\n\n*** Called get_completion on {messages}")
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        tools=[
            {
                "type": "function",
                "function": {
                    "name": "get_stock_quote",
                    "description": """Fetch the latest stock quote for ``symbol`` using the Alpha Vantage API.""",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "symbol": {
                                "type": "string",
                            },
                        },
                        "required": ["symbol"],
                    },
                },
            },
        ],
        temperature=1,
    )

    return response

In [17]:
import json

def answer_question(question, system_prompt="You are a helpful Financial Advisor."):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",  "content": question},
    ]
    loop_count_limit = 5
    loop_count = 0

    while True:
        loop_count += 1
        if loop_count > loop_count_limit:
            raise Exception("Loop count exceeded")

        response = get_completion(AI_MODEL, messages)
        print(f"\n\n*** For input {messages}\n*** Received response {response}\n\n")

        choice = response.choices[0]
        msg = choice.message
        finish = choice.finish_reason

        # ---- Normal completion
        if finish == "stop":
            return msg.content

        # ---- Modern tools API (parallel tool calls)
        if finish == "tool_calls" and getattr(msg, "tool_calls", None):
            tool_calls = msg.tool_calls

            # Echo back assistant tool_calls (as dicts, not objects)
            messages.append({
                "role": "assistant",
                "content": None,
                "tool_calls": [
                    {
                        "id": tc.id,
                        "type": "function",
                        "function": {
                            "name": tc.function.name,
                            "arguments": tc.function.arguments or "{}",
                        },
                    } for tc in tool_calls
                ]
            })

            # Execute each tool call and append the tool results
            for tc in tool_calls:
                fn_name = tc.function.name
                arg_json = tc.function.arguments or "{}"
                try:
                    args = json.loads(arg_json) if isinstance(arg_json, str) else (arg_json or {})
                except Exception:
                    args = {}

                # Prefer a single 'symbol' (case-insensitive); else pass kwargs
                if isinstance(args, dict):
                    symbol = args.get("symbol")
                    if symbol is None:
                        # try uppercase or other casings
                        for k in list(args.keys()):
                            if k.lower() == "symbol":
                                symbol = args[k]
                                break
                    if symbol is not None:
                        result = globals()[fn_name](symbol)
                    else:
                        result = globals()[fn_name](**args)
                else:
                    # args was not a dict → pass-through
                    result = globals()[fn_name](args)

                print(f"*** Called function {fn_name} with parameters {args}, returning {result}")

                messages.append({
                    "tool_call_id": tc.id,
                    "role": "tool",
                    "name": fn_name,
                    "content": json.dumps(result) if not isinstance(result, str) else result,
                })

            continue

        # ---- Legacy single function_call
        if finish == "function_call" and getattr(msg, "function_call", None):
            fn_name = msg.function_call.name
            arg_json = msg.function_call.arguments or "{}"
            try:
                args = json.loads(arg_json) if isinstance(arg_json, str) else (arg_json or {})
            except Exception:
                args = {}

            if isinstance(args, dict):
                symbol = args.get("symbol")
                if symbol is None:
                    for k in list(args.keys()):
                        if k.lower() == "symbol":
                            symbol = args[k]
                            break
                if symbol is not None:
                    result = globals()[fn_name](symbol)
                else:
                    result = globals()[fn_name](**args)
            else:
                result = globals()[fn_name](args)

            print(f"*** Called function {fn_name} with parameters {args}, returning {result}")

            # Echo assistant function_call (legacy)
            messages.append({
                "role": "assistant",
                "content": None,
                "function_call": {"name": fn_name, "arguments": arg_json},
            })
            # Then function result (legacy)
            messages.append({
                "role": "function",
                "name": fn_name,
                "content": json.dumps(result) if not isinstance(result, str) else result,
            })
            continue

        # Fallback: just return whatever text we got
        return msg.content or ""

In [18]:
q="How is Nvidia's stock today?"
a=answer_question(q,"You are a helpful Financial Planner.")
print(f"\n\nUser Question: {q}\nOpenAI Answer: {a}")



*** Called get_completion on [{'role': 'system', 'content': 'You are a helpful Financial Planner.'}, {'role': 'user', 'content': "How is Nvidia's stock today?"}]


*** For input [{'role': 'system', 'content': 'You are a helpful Financial Planner.'}, {'role': 'user', 'content': "How is Nvidia's stock today?"}]
*** Received response ChatCompletion(id='chatcmpl-DXJ8CQG81NgILLE9Sg3HJqJUSJIWp', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_63aNS1YQjspqost0hY7EQjNv', function=Function(arguments='{"symbol":"NVDA"}', name='get_stock_quote'), type='function')]))], created=1776830736, model='gpt-5-mini-2025-08-07', object='chat.completion', service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=89, prompt_tokens=152, total_tokens=241, completion_tokens_detail

# GenAI with direct LLM-to-MCP connection

In [20]:
from openai import OpenAI

client = OpenAI()

response = client.responses.create(
    model="gpt-5-mini",
    input="Get the latest stock quote for AAPL using the Alpha Vantage MCP server. It is okay to get delayed or most recent close.",
    tools=[
        {
            "type": "mcp",
            "server_url": f"https://mcp.alphavantage.co/mcp?apikey={ALPHAVANTAGE_API_KEY}",
            "server_label": "alphavantage",
            "require_approval": "never"
        }
    ]
)

print(response.output_text)

Here’s the latest Alpha Vantage GLOBAL_QUOTE for AAPL (latest trading day shown):

- Symbol: AAPL
- Latest trading day: 2026-04-21
- Price: $266.17
- Previous close: $273.05
- Change: -$6.88 (-2.5197%)
- Open: $271.50
- High: $272.80
- Low: $265.40
- Volume: 50,209,755

Source: Alpha Vantage GLOBAL_QUOTE (most recent available quote). Want intraday ticks or a CSV/JSON export?


# What does the MCP server call return?

In [26]:
import requests
from IPython.display import display, JSON, Markdown

url = f"https://mcp.alphavantage.co/mcp?apikey={ALPHAVANTAGE_API_KEY}"
payload = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "tools/list"
}
response = requests.post(url, json=payload)
# print(response.json())

data = response.json()
# If it's a string, parse it
if isinstance(data, str):
    data = json.loads(data)


display(JSON(data))

<IPython.core.display.JSON object>

In [32]:
import requests
import json
from IPython.display import display, Markdown

url = f"https://mcp.alphavantage.co/mcp?apikey={ALPHAVANTAGE_API_KEY}"
payload = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "tools/list"
}

response = requests.post(url, json=payload)
data = response.json()

if isinstance(data, str):
    data = json.loads(data)

tools = data["result"]["tools"]

md = "## 📦 Alpha Vantage MCP Tools\n\n"

for tool in tools:
    md += f"### 🔧 `{tool['name']}`\n\n"
    md += f"{tool['description'].strip()}\n\n"
    props = tool["inputSchema"]["properties"]
    required = tool["inputSchema"].get("required", [])
    if props:
        md += "**Parameters:**\n"
        for k, v in props.items():
            req = " (required)" if k in required else ""
            md += f"- `{k}` ({v['type']}){req}: {v.get('description','')}\n"
        md += "\n"
    else:
        md += "_No parameters_\n\n"

display(Markdown(md))

## 📦 Alpha Vantage MCP Tools

### 🔧 `TOOL_LIST`

List all available Alpha Vantage API tools with their names and descriptions. IMPORTANT: This returns only tool names and descriptions, NOT parameter schemas. You MUST call TOOL_GET(tool_name) to retrieve the full inputSchema (required parameters, types, descriptions) before calling TOOL_CALL. Calling TOOL_CALL without first calling TOOL_GET will fail because you won't know the required parameters. Workflow: TOOL_LIST -> TOOL_GET(tool_name) -> TOOL_CALL(tool_name, arguments)

_No parameters_

### 🔧 `TOOL_GET`

Get the full schema for one or more tools including all parameters. After discovering tools via TOOL_LIST, use this to get the complete parameter schema before calling the tool. You can provide either a single tool name or a list of tool names if you're unsure which one to use.

**Parameters:**
- `tool_name` (string) (required): The name of the tool to get schema for (e.g., "TIME_SERIES_DAILY"),

### 🔧 `TOOL_CALL`

Execute a tool by name with the provided arguments. IMPORTANT: You MUST call TOOL_GET(tool_name) first to retrieve the full parameter schema before calling this tool. The arguments must match the schema returned by TOOL_GET, including all required parameters. Calling without the correct arguments will result in errors. Workflow: TOOL_LIST -> TOOL_GET(tool_name) -> TOOL_CALL(tool_name, arguments)

**Parameters:**
- `tool_name` (string) (required): The name of the tool to call (e.g., "TIME_SERIES_DAILY")
- `arguments` (string) (required): Dictionary of arguments matching the tool's parameter schema from TOOL_GET



In [31]:
import requests
import json
import ast
from IPython.display import display, JSON, Markdown

url = f"https://mcp.alphavantage.co/mcp?apikey={ALPHAVANTAGE_API_KEY}"

payload = {
    "jsonrpc": "2.0",
    "id": 2,
    "method": "tools/call",
    "params": {
        "name": "TOOL_GET",
        "arguments": {
            "tool_name": "GLOBAL_QUOTE"
        }
    }
}

resp = requests.post(url, json=payload)
#print(resp.status_code)
#print(resp.text)

outer = resp.json()
inner_text = outer["result"]["content"][0]["text"]
data = ast.literal_eval(inner_text)

md = f"""
### {data['name']}
{data['description'].strip()}
**Parameters**
"""
for k, v in data["parameters"]["properties"].items():
    req = " (required)" if k in data["parameters"]["required"] else ""
    md += f"- **{k}** ({v['type']}){req}: {v['description']}\n"
display(Markdown(md))


### GLOBAL_QUOTE
Returns the latest price and volume information for a ticker.

Args:
    symbol: The symbol of the global ticker. For example: symbol=IBM
    datatype: By default, datatype=csv. Strings json and csv are accepted with the following specifications:
             json returns the data in JSON format; csv returns the data as a CSV (comma separated value) file.


        entitlement: "delayed" for 15-minute delayed data, "realtime" for realtime data
Returns:
    Dict or string containing the latest quote information based on datatype parameter.
**Parameters**
- **symbol** (string) (required): The symbol of the global ticker. For example: symbol=IBM
- **datatype** (string): By default, datatype=csv. Strings json and csv are accepted with the following specifications:
- **entitlement** (string): "delayed" for 15-minute delayed data, "realtime" for realtime data
